# Step 3: Preprocess (Tiền Xử Lý & Làm Sạch Dữ Liệu)
### Theo Quy Trình 8 Bước Từ Slide Bài Giảng Chapter 4 (NEU FIT)

---

### 1. Phân Tích Chuyên Sâu Cách Làm:
Slide 21 đặt mục tiêu: *Output of this step: a table of numbers only, with no duplicates and no missing values.*
Các tác vụ cốt lõi:
1. **Drop duplicate rows:** Loại bỏ các bản ghi trùng khóa thời gian và địa điểm.
2. **Turn wrong values into corrected/valid values:**
   - Nhiệt độ độ F $	o$ độ C ($T > 45^\circ	ext{C}$).
   - Doanh thu gõ thừa số 0 ($> 1000$ triệu $	o$ chia $1000$).
   - Lệch dấu phẩy thập phân ($10	imes$ và $0.1	imes$) dựa trên tỷ số so với trung vị thứ trong tuần của quán.
3. **Unify different spellings:** Chuẩn hóa `District 10`, `District 3`, `District 7`.
4. **Handle missing values:**
   - Điền khuyết diện tích `area_m2` theo tỷ lệ trung vị diện tích/ghế theo loại quán.
   - Nội suy chuỗi thời tiết liên tục 2024 - 2026.
5. **Encode text columns as numbers:** One-hot encoding `pd.get_dummies` cho `city`, `store_type`, `tet_phase`.

### 2. Dùng Những Gì Để Làm?
- **Thư viện:** `pandas`, `numpy`.
- **Hàm:** `drop_duplicates()`, `replace()`, `fillna()`, `get_dummies(dtype=int)`.

In [1]:
import os, sys
import pandas as pd
import numpy as np

for p in [os.path.abspath(".."), os.path.abspath("."), os.path.abspath("../..")]:
    if os.path.exists(os.path.join(p, "data")):
        DATA_DIR = os.path.join(p, "data")
    if p not in sys.path and os.path.exists(os.path.join(p, "src")):
        sys.path.insert(0, p)

from src.cleaning import clean_stores, clean_weather, clean_holidays, clean_train, clean_test

# 1. Đọc dữ liệu
train_raw = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_raw = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
stores_raw = pd.read_csv(os.path.join(DATA_DIR, "stores.csv"))
weather_raw = pd.read_csv(os.path.join(DATA_DIR, "weather.csv"))
holidays_raw = pd.read_csv(os.path.join(DATA_DIR, "holidays.csv"))

# 2. Áp dụng pipeline làm sạch
stores_c = clean_stores(stores_raw)
weather_c = clean_weather(weather_raw, stores_c["city"].unique())
nat_hol, evt_hol, tet_dates = clean_holidays(holidays_raw)
train_c, stats_train = clean_train(train_raw, stores_c)
test_c, stats_test = clean_test(test_raw, stores_c)

print("KẾT QUẢ TIỀN XỬ LÝ LÀM SẠCH:")
print(f"- Loại bỏ trùng lặp train : {stats_train['duplicates_removed']} dòng")
print(f"- Sửa lỗi gõ thừa 1000x  : {stats_train['fixed_1000x']} dòng")
print(f"- Sửa lỗi lệch thập phân 10x: {stats_train['fixed_10x']} dòng")
print(f"- Sửa lỗi lệch thập phân 0.1x: {stats_train['fixed_01x']} dòng")
print(f"- Train sạch sau cùng     : {train_c.shape}")
print(f"- Test sạch sau cùng      : {test_c.shape}")

KẾT QUẢ TIỀN XỬ LÝ LÀM SẠCH:
- Loại bỏ trùng lặp train : 132 dòng
- Sửa lỗi gõ thừa 1000x  : 30 dòng
- Sửa lỗi lệch thập phân 10x: 53 dòng
- Sửa lỗi lệch thập phân 0.1x: 26 dòng
- Train sạch sau cùng     : (26500, 4)
- Test sạch sau cùng      : (3869, 4)


### 3. Mã Hóa One-Hot Encoding (Text Columns to Numbers)
Theo Slide 25: Chuyển toàn bộ các biến phân loại (`store_type`, `city`) thành các cột nhị phân $0/1$ để mô hình có thể tính toán ma trận.

In [2]:
# Ghép thông tin cửa hàng vào train sạch
df_encoded = train_c.merge(stores_c[["store_id", "store_type", "city", "seats", "area_m2", "has_parking"]], on="store_id", how="left")
df_encoded["dow"] = df_encoded["date"].dt.dayofweek.astype(str)

# Áp dụng get_dummies
df_numeric = pd.get_dummies(df_encoded, columns=["store_type", "city", "dow"], dtype=int)
print(f"Kích thước bảng toàn số sau One-Hot Encoding: {df_numeric.shape}")
print("\nDanh sách các cột:")
print(df_numeric.columns.tolist()[:15], "... [tổng cộng", len(df_numeric.columns), "cột]")
display(df_numeric.head(3))

print(">>> HOÀN THÀNH STEP 3: DỮ LIỆU ĐÃ SẠCH 100% VÀ SẴN SÀNG CHO MÔ HÌNH HÓA!")

Kích thước bảng toàn số sau One-Hot Encoding: (26500, 21)

Danh sách các cột:
['date', 'store_id', 'is_promo', 'revenue', 'seats', 'area_m2', 'has_parking', 'store_type_mall', 'store_type_office', 'store_type_residential', 'store_type_street', 'store_type_university', 'city_Ha Noi', 'city_Ho Chi Minh', 'dow_0'] ... [tổng cộng 21 cột]


,date,store_id,is_promo,revenue,seats,area_m2,has_parking,store_type_mall,store_type_office,store_type_residential,...,store_type_university,city_Ha Noi,city_Ho Chi Minh,dow_0,dow_1,dow_2,dow_3,dow_4,dow_5,dow_6
0,2024-01-01,S02,0,6.44,28,95.0,1,0,0,1,...,0,1,0,1,0,0,0,0,0,0
1,2024-01-01,S03,0,8.86,38,120.0,0,0,1,0,...,0,1,0,1,0,0,0,0,0,0
2,2024-01-01,S04,0,17.08,41,101.0,0,0,0,0,...,0,1,0,1,0,0,0,0,0,0


>>> HOÀN THÀNH STEP 3: DỮ LIỆU ĐÃ SẠCH 100% VÀ SẴN SÀNG CHO MÔ HÌNH HÓA!
